# Lab 05 (cluster): Evaluate AI systems and choose a model

**Chapter 4, "Evaluate AI Systems"** (sections *Evaluation Criteria*, *Model Selection*, *Design Your Evaluation Pipeline*)
**Environment:** Docker JupyterLab on the KU VPN | **Time:** Part A about 2.5 hours (graded). Part B is an optional stretch for bonus points.
**This is a Part-A-only week** because Side Project 1 is due. Do Part B only if you have time.

Larkspur has to pick **one** of the four course chat models for the help-desk assistant. Chapter 4 says not to pick from a leaderboard: decide what matters for *your* application, build an evaluation pipeline on *your* data, and let the numbers decide. That is this lab:

1. design evaluation criteria for the assistant,
2. run all four models on 40 **gold questions** (questions with a checked, correct reference answer) with the documents provided,
3. measure correctness, abstention, latency, and output length,
4. dig into the **trap questions**, where the context contains two documents that disagree,
5. build a weighted scorecard and choose a model.

**Before you start**
- VPN on, Docker lab running.
- The main run sends about **290 requests** and takes roughly 20-30 minutes (the dense model is slow). Everything is cached in `lab05_cache.json`, so if your connection drops you can just re-run; finished requests are not repeated.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Evaluation set, gold questions** | A fixed list of questions, each with a checked, correct **reference answer**. "Gold" means the reference is trusted as the truth. | Every model answers the same questions, so the comparison is fair. |
| **Context** | The documents placed in the prompt next to the question (from Lab 01: everything the model reads for one answer). | The model should answer only from these documents. |
| **Trap question** | A question whose context contains two documents that disagree, for example an old and a current version of a policy. | Real company documents go out of date; the assistant must pick the current one. |
| **Unanswerable question, abstention** | A question whose answer is not in the context. **Abstaining** means the model declines ("Not in the documents") instead of guessing. | A help-desk assistant that makes things up is worse than one that says "I don't know". |
| **Evaluation criteria, metric** | A criterion is something you care about (for example correctness or speed). A metric is the number you measure it with. | Deciding them before you see any results keeps the choice honest. |
| **LLM judge** | A model that grades another model's answers against the reference (from Lab 04). | It can judge answers that are worded differently from the reference, but it can be biased. |
| **Exact checks, regex** | Checks that compare text directly, with no model involved. A **regex** (regular expression) is a pattern for finding text, such as "a dollar sign followed by digits". | Cheap and repeatable, so use them where they work. |
| **Latency, p50 / p95** | Latency is the seconds a request takes (from Lab 02). **p50** is the median request; **p95** is the time 95% of requests beat, so it shows the slow ones users sometimes hit. | A help desk notices the slow answers, not just the typical one. |
| **Input / output tokens** | Tokens sent to the model (instructions, documents, question) and tokens it writes back (from Lab 01). | Hosted APIs charge per token, so they drive cost. |
| **Temperature 0** | A sampling setting that makes the model pick its most likely token each time (from Lab 03). | Results are (nearly) repeatable, which an evaluation needs. |
| **Cache** | A file (`lab05_cache.json`) where each finished reply is saved. | If your connection drops, re-running skips the requests already done. |
| **Weighted scorecard** | Each metric is rescaled to 0-1 and multiplied by how much you said it matters (its **weight**); the results are added up. | Turns many numbers into one ranking, using your priorities. |
| **Benchmark, leaderboard** | Public test sets and the rankings of models on them. | Q3 asks why they are a weak guide for choosing a model for one specific application. |
| **Build vs. buy** | Hosting a model yourself versus paying a company's API per token. | Part B compares the two on cost. |

## Setup

Run both cells. The first connects to the course gateway (from Lab 02) and lists its models. The second defines `call()`, a helper that sends one request and saves the reply in the **cache** file `lab05_cache.json`, so a re-run never repeats a finished request. It then loads the gold questions and their documents from the corpus (files ending in `.jsonl`, "JSON Lines": one record per line) and sends each model a one-word test message. Each printed line is a model, its reply, and how many seconds it took.

In [ ]:
import json, re, time, math, random
from collections import Counter

import numpy as np
import pandas as pd

from itec632 import get_client, MODELS, NO_THINK, strip_think
from itec632.grading import LabChecker
from itec632.corpus import corpus_dir

client = get_client()
ck = LabChecker("lab05", part="cluster")
CANDIDATES = {"qwen-moe": MODELS["main"], "qwen-dense": MODELS["dense"], "gemma-moe": MODELS["gemma"], "qwen-small": MODELS["small"]}
print("Gateway models:", [m.id for m in client.models.list().data])

In [ ]:
CACHE_FILE = "lab05_cache.json"
try:
    CACHE = json.load(open(CACHE_FILE))
except Exception:
    CACHE = {}
REQUESTS = {"n": 0}
MIN_GAP = 1.5
_last_call = [0.0]


def call(model, messages, max_tokens=200, schema=None, cache_key=None):
    """One chat request. Returns a dict: text, latency_s, out_tokens, in_tokens (cached by cache_key).

    Latency is wall-clock time for the whole (non-streamed) request, measured on the first run only.
    """
    if cache_key and cache_key in CACHE:
        return CACHE[cache_key]
    kwargs = dict(model=model, messages=messages, max_tokens=max_tokens, temperature=0.0, extra_body=NO_THINK)
    if schema is not None:
        kwargs["response_format"] = {"type": "json_schema", "json_schema": {"name": "result", "schema": schema}}
    result = {"text": "", "latency_s": None, "out_tokens": None, "in_tokens": None}
    for attempt in range(3):
        wait = MIN_GAP - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        try:
            _last_call[0] = time.time()
            REQUESTS["n"] += 1
            t0 = time.perf_counter()
            r = client.chat.completions.create(**kwargs)
            latency = time.perf_counter() - t0
            usage = getattr(r, "usage", None)
            result = {"text": strip_think(r.choices[0].message.content or ""), "latency_s": latency,
                      "out_tokens": getattr(usage, "completion_tokens", None),
                      "in_tokens": getattr(usage, "prompt_tokens", None)}
            break
        except Exception as e:
            msg = str(e)
            if "response_format" in kwargs and ("response_format" in msg or "schema" in msg or "400" in msg):
                kwargs.pop("response_format")
                continue
            print(f"   (attempt {attempt + 1} failed: {msg[:100]})")
            time.sleep(5 * (attempt + 1))
    if cache_key and result["text"]:
        CACHE[cache_key] = result
        json.dump(CACHE, open(CACHE_FILE, "w"))
    return result


EVAL = corpus_dir() / "eval"
GOLD = [json.loads(line) for line in open(EVAL / "gold_qa.jsonl")]
BY_ID = {q["qid"]: q for q in GOLD}
CTX = {}
for line in open(EVAL / "gold_contexts.jsonl"):
    rec = json.loads(line)
    CTX[rec["qid"]] = rec["passages"]


def is_unanswerable(q):
    return q["reference_answer"].startswith("Not in the documents")


def format_context(qid):
    parts = [f"[{p['doc_id']} v{p['version']}, effective {p['effective_date']}] {p['title']} - {p['section']}\n{p['text']}"
             for p in CTX.get(qid, [])]
    return "\n\n---\n\n".join(parts) if parts else "(no documents)"


for name, model in CANDIDATES.items():
    r = call(model, [{"role": "user", "content": "Reply with the single word: ready"}], max_tokens=10, cache_key=f"ping|{name}")
    print(f"{name:11} -> {r['text']!r}  ({r['latency_s'] if r['latency_s'] is None else round(r['latency_s'], 2)} s)")

# Part A (graded)

## A1. Design the evaluation criteria

Chapter 4 groups evaluation criteria into **domain-specific capability**, **generation capability** (for us mainly factual consistency with the documents), **instruction-following capability**, and **cost and latency**. In plain words, for this assistant:

- *domain-specific capability*: does it get help-desk questions right?
- *generation capability*: does what it writes stay true to the documents, without made-up details?
- *instruction-following capability*: does it do what the system prompt says (short answers, "Not in the documents" when it can't answer)?
- *cost and latency*: how much does each answer cost, and how long does it take?

Write down what each means *for the Larkspur help-desk assistant* before you see any results; deciding after you see the numbers is how people talk themselves into the model they already liked.

The pipeline below will produce these metrics for every model (pick from them):

| Metric | Meaning | Better |
|---|---|---|
| `judge_correct` | LLM-judge correctness on answerable questions (correct = 1, partly = 0.5) | higher |
| `fact_recall` | share of key facts from the reference (amounts, times, names, counts) that appear in the answer | higher |
| `trap_accuracy` | `judge_correct` on trap questions only | higher |
| `abstention_rate` | share of unanswerable questions where the model declined | higher |
| `false_abstention` | share of answerable questions where the model wrongly declined | lower |
| `format_ok` | share of answers that follow the format (at most 2 sentences, no more than 80 words) | higher |
| `latency_p50_s`, `latency_p95_s` | median / 95th percentile request time | lower |
| `out_tokens_mean` | mean output tokens per answer (a cost proxy) | lower |

A **percentile** comes from sorting all the request times: p50 is the middle one (the median) and p95 is the time that 95% of requests beat. A **proxy** is a stand-in measure: more output tokens means a higher bill. A share such as 0.75 means 75%.

**TODO:** fill in `CRITERIA`: for each of the four required criteria, a weight, the metric you'll use for it, and one sentence on why it matters at Larkspur. Weights must sum to 1. You may add more criteria (each needs a metric). A **weight** says how much a criterion counts: 0.40 means 40% of the final score.

In [ ]:
CRITERIA = {
    "domain_capability":     {"weight": None, "metric": "", "why": ""},   # TODO
    "generation_quality":    {"weight": None, "metric": "", "why": ""},   # TODO
    "instruction_following": {"weight": None, "metric": "", "why": ""},   # TODO
    "cost_latency":          {"weight": None, "metric": "", "why": ""},   # TODO
}

In [ ]:
METRICS = {"judge_correct": True, "fact_recall": True, "trap_accuracy": True, "abstention_rate": True,
           "false_abstention": False, "format_ok": True, "latency_p50_s": False, "latency_p95_s": False,
           "out_tokens_mean": False}          # metric -> higher_is_better
required = {"domain_capability", "generation_quality", "instruction_following", "cost_latency"}
weights_ok = all(isinstance(c.get("weight"), (int, float)) and c["weight"] >= 0 for c in CRITERIA.values())
ck.check("A1_criteria_complete", required <= set(CRITERIA) and weights_ok
         and all(c.get("metric") in METRICS and len(c.get("why", "")) >= 20 for c in CRITERIA.values()),
         points=2, hint="Every criterion needs a numeric weight, a metric from the METRICS table, and a 'why' sentence.")
ck.check("A1_weights_sum_to_1", weights_ok and abs(sum(c["weight"] for c in CRITERIA.values()) - 1) < 0.01,
         points=1, hint="Weights must add up to 1.0.")
ck.record("A1_weights", {k: c.get("weight") for k, c in CRITERIA.items()})

## A2. The evaluation set and its context

Chapter 4's pipeline advice: evaluate on data that looks like real use, include slices you care about, and keep the set fixed so models are compared on equal terms. Our 40 questions:

- **16 trap questions**: the gold context deliberately contains **two documents that disagree** (for example an old and a new version). The reference answer is the current truth. We are *not* telling the model or you which document is outdated.
- **16 ordinary answerable questions.**
- **8 unanswerable questions**: the context is related but does not contain the answer.

The 16 ordinary questions are picked at random, but with a fixed **seed** (the `5` in `random.Random(5)`), so everyone gets the same 16. Every model gets exactly the same system prompt and context. Look at one trap question's context below: what would *you* answer, and how did you decide?

In [ ]:
TRAP_IDS = ['Q063', 'Q019', 'Q030', 'Q057', 'Q047', 'Q079', 'Q058', 'Q006', 'Q028', 'Q011', 'Q039', 'Q022', 'Q062', 'Q035', 'Q078', 'Q021']
normal_pool = [q["qid"] for q in GOLD if not q["trap"] and q["category"] != "data" and not is_unanswerable(q)]
NORMAL_IDS = random.Random(5).sample(normal_pool, 16)
UNANS_IDS = [q["qid"] for q in GOLD if is_unanswerable(q)]
EVAL_IDS = TRAP_IDS + NORMAL_IDS + UNANS_IDS
print(len(EVAL_IDS), "questions;", Counter(BY_ID[i]["category"] for i in EVAL_IDS))

SYSTEM = ("You are the Larkspur Ridge IT help-desk assistant. Answer using ONLY the documents provided. "
          "Answer in one or two sentences with the key fact first. "
          "If the documents do not contain the answer, reply exactly: Not in the documents.")


def build_messages(qid):
    q = BY_ID[qid]
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Documents:\n{format_context(qid)}\n\nQuestion: {q['question']}"}]


example = TRAP_IDS[1]
print("QUESTION:", BY_ID[example]["question"], "\n")
print(format_context(example)[:2500])

**Reading the output:** the first line counts the 40 questions by category. Then comes one trap question and its context. Each document starts with a header in square brackets: the document ID, its version number (`v...`), and its **effective date** (the date it took effect), followed by the title and section. Documents are separated by `---`.

## A3. Cheap, exact checks first: key facts, abstention, format

Before paying for an LLM judge, use exact checks where they work (Chapter 3's *exact evaluation*: comparing text directly, with no model involved). Three helpers:

- `key_facts(reference)` pulls the checkable facts out of a reference answer: dollar amounts, percentages, clock times, phone numbers, counts with units, hardware models, and staff names. (Given.)
- `fact_recall(answer, facts)`: **TODO**, the share of those facts found in the answer.
- `is_abstention(answer)`: **TODO**, does the answer decline? Models phrase it many ways ("not mentioned", "the documents don't specify"). Add patterns until the test cases pass.

`key_facts()` and `is_abstention()` use **regular expressions** (regex): small patterns for finding text. For example `\$\d+` means "a dollar sign followed by one or more digits", and `(?:does|do)` means "does or do". Look at the patterns already in `FACT_PATTERNS` and `ABSTAIN_PATTERNS` for more; an AI assistant is good at explaining a regex line by line.

In [ ]:
STAFF = ["Dana Whitfield", "Marcus Oyelaran", "Priya Raman", "Tom Kessler", "Renee Castillo", "Jordan Pike", "Alicia Brandt",
         "Kevin Duong", "Hannah Moreau", "Samuel Ortiz", "Lena Fischer", "Andre Mitchell", "Grace Liu", "Brian Novak",
         "Maya Patel", "Chris Bauer", "Tasha Greene", "Luis Herrera", "Emily Sandoval", "Noah Whitaker", "Olivia Grant",
         "Derek Simmons", "Rachel Kim", "Isaac Brennan", "Greg Halvorsen"]
FACT_PATTERNS = [
    r"\$\d[\d,]*(?:\.\d+)?",                         # $247.50
    r"\b\d+(?:\.\d+)?%",                              # 20%
    r"\b\d{1,2}:\d{2}\s?[AP]M\b",                      # 7:00 AM
    r"\(\d{3}\) \d{3}-\d{4}",                          # (913) 555-0142
    r"\b\d+(?:\.\d+)?\s(?:business\s)?(?:days?|hours?|minutes?|GB|years?|weeks?)\b",   # 5 business days
    r"\b(?:Latitude|OptiPlex(?: Micro)?)\s\d{4}\b",     # Latitude 5550
]


def _norm(s):
    return re.sub(r"\s+", " ", (s or "").lower().replace(",", "").replace("$", "")).strip()


def key_facts(reference):
    """Checkable facts in a reference answer (normalized strings). Empty list if none."""
    if reference.startswith("Not in the documents"):
        return []
    facts = []
    for pat in FACT_PATTERNS:
        facts += re.findall(pat, reference)
    facts += [n for n in STAFF if n in reference]
    out = []
    for f in facts:
        f = _norm(f).replace(".00", "")
        if f and f not in out:
            out.append(f)
    return out


for qid in EVAL_IDS[:4]:
    print(qid, key_facts(BY_ID[qid]["reference_answer"]), "<-", BY_ID[qid]["reference_answer"][:70])

**Reading the output:** each line is a question ID, the list of facts `key_facts()` found in its reference answer, and the start of that reference answer. Facts are lowercased with `$` and commas removed, so `$1,200` and `1200` count as the same fact. An empty list `[]` means there was nothing checkable.

In [ ]:
def fact_recall(answer, facts):
    """Share of `facts` (from key_facts) that appear in `answer`. None if there are no facts to check."""
    if not facts:
        return None
    a = _norm(answer).replace(".00", "")
    # TODO: return the fraction of facts f for which (f in a) is True
    return None


ABSTAIN_PATTERNS = [
    r"not in the (?:provided )?documents?",
    # TODO: add regex patterns (lowercase) so every abstention test below is caught and no non-abstention is
]


def is_abstention(answer):
    a = (answer or "").lower()
    return any(re.search(p, a) for p in ABSTAIN_PATTERNS)

In [ ]:
abstain_tests = {
    "Not in the documents.": True,
    "The documents do not specify the guest Wi-Fi password.": True,
    "That information is not mentioned in the provided documents.": True,
    "I can't find a 401(k) match in these documents.": True,
    "The documents don't say who replaced Greg.": True,
    "Essentials P2 response is 2 business hours.": False,
    "Call PagerLine at (913) 555-0142; it pages the on-call engineer.": False,
    "No. SMS codes are not allowed for new enrollments.": False,
}
abstain_ok = all(is_abstention(t) == want for t, want in abstain_tests.items())
for t, want in abstain_tests.items():
    if is_abstention(t) != want:
        print("WRONG:", repr(t), "expected", want)
fr = fact_recall("It is $247.50 per hour with a 1 hour minimum", ["247.50", "1 hour"])
ck.check("A3_fact_recall", fr is not None and abs(fr - 1.0) < 1e-9
         and abs(fact_recall("It is $210 per hour", ["247.50", "1 hour"]) - 0.0) < 1e-9
         and fact_recall("anything", []) is None, points=2,
         hint="fact_recall should be 1.0 when all facts appear, 0.0 when none do, None when there are no facts.")
ck.check("A3_is_abstention", abstain_ok, points=2, hint="Add patterns to ABSTAIN_PATTERNS until every test case is classified correctly.")

## A4. Run all four models

Each model answers all 40 questions (160 requests). We keep the text, the latency, and the token counts. **Do not change the prompt between models**; if you change it, re-run all four.

`format_ok` is computed here: at most 2 sentences and at most 80 words.

In [ ]:
def count_sentences(text):
    return len([s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z])", (text or "").strip()) if s])


RUNS = []
t_start = time.time()
for name, model in CANDIDATES.items():
    t0 = time.time()
    for qid in EVAL_IDS:
        r = call(model, build_messages(qid), max_tokens=200, cache_key=f"ans|{name}|{qid}")
        RUNS.append({"model": name, "qid": qid, **r})
    print(f"{name:11} done in {time.time() - t0:.0f}s")
runs = pd.DataFrame(RUNS)
q_meta = pd.DataFrame([{"qid": i, "trap": BY_ID[i]["trap"], "unanswerable": is_unanswerable(BY_ID[i]),
                        "category": BY_ID[i]["category"]} for i in EVAL_IDS])
runs = runs.merge(q_meta, on="qid")
runs["abstained"] = runs["text"].apply(is_abstention)
runs["format_ok"] = runs["text"].apply(lambda t: bool(t) and count_sentences(t) <= 2 and len(t.split()) <= 80)
runs["facts"] = runs["qid"].apply(lambda i: key_facts(BY_ID[i]["reference_answer"]))
runs["fact_recall"] = [fact_recall(t, f) for t, f in zip(runs["text"], runs["facts"])]
print(f"total {time.time() - t_start:.0f}s; empty answers:", int((runs["text"] == "").sum()))
runs[["model", "qid", "latency_s", "out_tokens", "abstained", "fact_recall", "text"]].head(8)

**Reading the table:** the first 8 of 160 rows, one per model and question. `latency_s` is the seconds that request took, `out_tokens` the length of the answer in tokens, `abstained` whether `is_abstention()` thinks the model declined, `fact_recall` the share of key facts it included (empty, shown as `NaN` or `None`, when the question has no checkable facts), and `text` is the answer itself. Replies come from the cache on a re-run, so the latencies are always from your first run.

In [ ]:
ck.check("A4_all_models_answered", len(runs) == 160 and (runs["text"] != "").sum() >= 150, points=3,
         hint="You need 4 models x 40 answers with (almost) no empty replies. Re-run the cell; finished calls are cached.")
ck.check("A4_latency_measured", runs["latency_s"].notna().sum() >= 150, points=1, hint="Latency should be recorded for each answer.")

## A5. LLM judge for correctness

Fact recall misses correct answers phrased differently ("two business days") and cannot judge procedures. So `qwen-moe` judges every answer to an **answerable** question against the reference (128 requests). It also flags answers that **contradict** the reference, that is, state a specific different fact; on trap questions that almost always means the model followed the other document.

Remember Lab 4: `qwen-moe` judging `qwen-moe`'s own answers may be biased. Keep that in mind when you read the table. `VERDICT_SCHEMA` is a **JSON schema**: a description of the exact shape the judge's reply must have (Lab 06 covers these). **Parsing** means turning the judge's text reply into Python values. Parsing is defensive: unparseable judgments become `None` and are counted, not silently dropped.

The first cell ends by testing `parse_verdict()` on five sample judge replies: each printed line is a reply, then the `(verdict, contradicts)` pair it extracted (`None` means it couldn't tell).

In [ ]:
VERDICT_SCHEMA = {"type": "object", "properties": {
    "verdict": {"type": "string", "enum": ["correct", "partially_correct", "incorrect", "abstained"]},
    "contradicts_reference": {"type": "boolean"},
    "reason": {"type": "string"}}, "required": ["verdict", "contradicts_reference", "reason"]}

JUDGE_PROMPT = """You are checking an answer from Larkspur Ridge IT's help-desk assistant.
The REFERENCE ANSWER is correct and current. Compare the ASSISTANT ANSWER to it.
- verdict: "correct" if it states the reference's key facts with nothing conflicting; "partially_correct" if the main fact is right
  but something important is missing; "incorrect" if the main fact is wrong or missing; "abstained" if it says it cannot answer.
- contradicts_reference: true if the answer states a specific fact (number, time, name, rule, or procedure step) that conflicts
  with the reference, otherwise false.

QUESTION: {question}
REFERENCE ANSWER: {reference}
ASSISTANT ANSWER: {answer}

Reply with JSON only: {"verdict": "...", "contradicts_reference": true or false, "reason": "<one sentence>"}"""
VERDICT_SCORE = {"correct": 1.0, "partially_correct": 0.5, "incorrect": 0.0, "abstained": 0.0}


def parse_verdict(text):
    """Return (verdict or None, contradicts True/False/None) from a judge reply, tolerating messy output."""
    if not text:
        return None, None
    verdict, contra = None, None
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            d = json.loads(m.group(0))
            v = str(d.get("verdict", "")).strip().lower().replace(" ", "_")
            verdict = v if v in VERDICT_SCORE else None
            c = d.get("contradicts_reference")
            contra = c if isinstance(c, bool) else ({"true": True, "yes": True, "false": False, "no": False}.get(str(c).lower()))
        except Exception:
            pass
    low = text.lower()
    if verdict is None:
        for v in ["partially_correct", "partially correct", "incorrect", "abstained", "correct"]:
            if re.search(r"\b" + v + r"\b", low):
                verdict = v.replace(" ", "_")
                break
    if contra is None:
        mc = re.search(r"contradicts?_?(?:reference)?\W{0,5}(true|false|yes|no)", low)
        contra = None if not mc else mc.group(1) in ("true", "yes")
    return verdict, contra


for t in ['{"verdict": "correct", "contradicts_reference": false, "reason": "ok"}', "Verdict: INCORRECT. contradicts: yes",
          "partially correct", "", "no idea"]:
    print(repr(t)[:60], "->", parse_verdict(t))

In [ ]:
JUDGE = MODELS["main"]
judged = []
answerable_runs = runs[~runs["unanswerable"]]
t0 = time.time()
for _, row in answerable_runs.iterrows():
    q = BY_ID[row["qid"]]
    prompt = (JUDGE_PROMPT.replace("{question}", q["question"]).replace("{reference}", q["reference_answer"])
              .replace("{answer}", row["text"] or "(empty answer)"))
    r = call(JUDGE, [{"role": "user", "content": prompt}], max_tokens=150, schema=VERDICT_SCHEMA,
             cache_key=f"judge|{row['model']}|{row['qid']}")
    v, c = parse_verdict(r["text"])
    judged.append({"model": row["model"], "qid": row["qid"], "verdict": v, "contradicts": c,
                   "judge_score": VERDICT_SCORE.get(v) if v else None})
judged = pd.DataFrame(judged)
runs = runs.drop(columns=[c for c in ["verdict", "contradicts", "judge_score"] if c in runs.columns]).merge(
    judged, on=["model", "qid"], how="left")
print(f"judged {len(judged)} answers in {time.time() - t0:.0f}s; unparseable verdicts:", int(judged["verdict"].isna().sum()))

In [ ]:
ck.check("A5_judged", len(judged) == 128 and judged["verdict"].notna().sum() >= 100, points=2,
         hint="Expected 128 judgments (4 models x 32 answerable questions) and most of them parsed.")

## A6. The comparison table

**TODO:** complete `summarize()` so it returns one row of metrics for a model's runs. Use `safe_mean()` (it ignores `None`/NaN and returns `None` for an empty list instead of dividing by zero). The answerable/unanswerable/trap subsets are prepared for you.

In [ ]:
def safe_mean(values):
    v = pd.to_numeric(pd.Series(list(values), dtype="object"), errors="coerce").dropna()
    return float(v.mean()) if len(v) else None


def pctl(values, p):
    v = pd.to_numeric(pd.Series(list(values), dtype="object"), errors="coerce").dropna()
    return float(np.percentile(v, p)) if len(v) else None

In [ ]:
def summarize(d):
    ans, una, trap = d[~d["unanswerable"]], d[d["unanswerable"]], d[d["trap"]]
    return {
        "judge_correct": safe_mean(ans["judge_score"]),
        "fact_recall": None,        # TODO: safe_mean of ans["fact_recall"]
        "trap_accuracy": None,      # TODO: safe_mean of the judge scores on trap questions
        "abstention_rate": None,    # TODO: safe_mean of una["abstained"]
        "false_abstention": None,   # TODO: safe_mean of ans["abstained"]
        "format_ok": safe_mean(d["format_ok"]),
        "latency_p50_s": pctl(d["latency_s"], 50),
        "latency_p95_s": None,      # TODO: 95th percentile of latency (use pctl)
        "out_tokens_mean": safe_mean(d["out_tokens"]),
    }

In [ ]:
comparison = pd.DataFrame({m: summarize(runs[runs["model"] == m]) for m in CANDIDATES}).T
display(comparison.astype(float).round(3))
filled = comparison.notna().sum().sum()
ck.check("A6_comparison_table", comparison.shape == (4, len(METRICS)) and filled >= 4 * len(METRICS) - 2, points=3,
         hint="Every metric should have a value for every model. Finish the TODOs in summarize().")
for m in CANDIDATES:
    for k in ["judge_correct", "trap_accuracy", "abstention_rate", "latency_p50_s"]:
        v = comparison.loc[m, k]
        ck.record(f"A6_{m}_{k}", None if v is None or pd.isna(v) else float(v))

**Reading the table:** one row per model and one column per metric from the A1 table. Shares are between 0 and 1 (0.75 = 75%), latencies are in seconds, and `out_tokens_mean` is tokens per answer. The A1 table says whether higher or lower is better for each column.

## A7. Trap-question breakdown

On the trap questions, the context had two documents that disagree. For each model, how often did it give the **current** answer, how often did it give an answer that **contradicts** the reference (likely following the other document), and how often did it decline?

**TODO:** complete `trap_breakdown()`: return the share of trap answers that are (a) judged correct, (b) flagged `contradicts`, (c) abstained.

In [ ]:
def trap_breakdown(d):
    t = d[d["trap"]]
    n = len(t)
    if n == 0:
        return {"current": None, "contradicts_reference": None, "abstained": None, "n": 0}
    # TODO: shares (out of n) with verdict == "correct", contradicts == True, and abstained == True
    return {"current": None, "contradicts_reference": None, "abstained": None, "n": n}

In [ ]:
traps = pd.DataFrame({m: trap_breakdown(runs[runs["model"] == m]) for m in CANDIDATES}).T
display(traps)
nontrap = {m: safe_mean(runs[(runs["model"] == m) & ~runs["trap"] & ~runs["unanswerable"]]["judge_score"]) for m in CANDIDATES}
print("judge_correct on ordinary (non-trap) answerable questions:", {m: None if v is None else round(v, 2) for m, v in nontrap.items()})
ck.check("A7_trap_breakdown", traps["current"].notna().all() and traps["contradicts_reference"].notna().all(), points=2,
         hint="trap_breakdown() should return three shares between 0 and 1 for each model.")
for m in CANDIDATES:
    v = traps.loc[m, "contradicts_reference"]
    ck.record(f"A7_{m}_trap_contradicts", None if v is None or pd.isna(v) else float(v))

**Reading the table:** one row per model, over its `n` = 16 trap answers. `current` is the share the judge marked fully correct (it gave the current answer), `contradicts_reference` the share that stated a different specific fact (usually taken from the outdated document), and `abstained` the share where it declined. They don't have to add up to 1: an answer can be partly correct, or wrong without contradicting anything. The line printed under the table is the judge score on the ordinary (non-trap) questions, for comparison.

Now look at the cases behind the numbers. For each trap question a model got wrong, the cell prints the answer and the **headers** of the documents in its context (doc id, version, effective date). Read two or three of them against the documents. Which document did the model follow? What in the context *could* have told it which one to trust?

In [ ]:
wrong = runs[runs["trap"] & (runs["contradicts"] == True)].sort_values("qid")
print(len(wrong), "trap answers flagged as contradicting the reference\n")
for qid, grp in list(wrong.groupby("qid"))[:6]:
    print(f"{qid}: {BY_ID[qid]['question']}")
    print("   reference:", BY_ID[qid]["reference_answer"])
    for p in CTX[qid]:
        print(f"   context doc: {p['doc_id']} v{p['version']} (effective {p['effective_date']}) - {p['section'][:50]}")
    for _, r in grp.iterrows():
        print(f"   {r['model']:11} said: {r['text'][:160]}")
    print()

## A8. Weighted scorecard and your choice

To combine metrics with different units, rescale each one across the four models to 0-1 (best model = 1, worst = 0), flipping metrics where lower is better. Then each model's score is the weighted sum over your `CRITERIA`.

**TODO:** complete `rescale()`. If all four models have the same value (max equals min), give everyone 1.0 so you never divide by zero. Missing values get 0.

In [ ]:
def rescale(values, higher_is_better=True):
    """values: dict model -> number (or None). Returns dict model -> 0..1."""
    nums = {m: v for m, v in values.items() if v is not None and not pd.isna(v)}
    if not nums:
        return {m: 0.0 for m in values}
    lo, hi = min(nums.values()), max(nums.values())
    # TODO: for each model: 0.0 if missing; 1.0 if hi == lo; else (v - lo) / (hi - lo), flipped (1 - s) if not higher_is_better
    return {m: 0.0 for m in values}

In [ ]:
score_rows = {}
for crit, c in CRITERIA.items():
    metric = c.get("metric")
    if metric in METRICS:
        scaled = rescale({m: comparison.loc[m, metric] for m in CANDIDATES}, METRICS[metric])
        score_rows[f"{crit} ({metric})"] = scaled
scorecard = pd.DataFrame(score_rows)
weights = pd.Series({f"{k} ({c.get('metric')})": (c.get("weight") or 0) for k, c in CRITERIA.items() if c.get("metric") in METRICS})
if len(scorecard.columns):
    scorecard["WEIGHTED_TOTAL"] = (scorecard[weights.index] * weights).sum(axis=1)
    display(scorecard.round(3).sort_values("WEIGHTED_TOTAL", ascending=False))

**Reading the scorecard:** one row per model, best first. Each column is one of your criteria (with its metric in brackets), rescaled so the best of the four models gets 1 and the worst gets 0, already flipped where lower is better. `WEIGHTED_TOTAL` multiplies each column by your weight and adds them up. Rescaling stretches small gaps: a 1 versus 0 can hide a tiny real difference, so check the raw numbers in the A6 table before you decide.

In [ ]:
CHOSEN_MODEL = ""   # TODO: the model you pick for the help-desk assistant ("qwen-moe", "qwen-dense", "gemma-moe" or "qwen-small")

In [ ]:
t = rescale({"a": 1.0, "b": 3.0, "c": None, "d": 2.0})
t2 = rescale({"a": 5.0, "b": 5.0}, higher_is_better=False)
ck.check("A8_rescale", abs(t["a"] - 0) < 1e-9 and abs(t["b"] - 1) < 1e-9 and t["c"] == 0.0 and abs(t["d"] - 0.5) < 1e-9
         and t2 == {"a": 1.0, "b": 1.0} and abs(rescale({"a": 1.0, "b": 3.0}, False)["a"] - 1) < 1e-9, points=2,
         hint="Best -> 1, worst -> 0, flip when lower is better, missing -> 0, all equal -> 1.0.")
ck.check("A8_model_chosen", CHOSEN_MODEL in CANDIDATES, points=1, hint="Set CHOSEN_MODEL to one of the four model names.")
ck.record("A8_chosen_model", CHOSEN_MODEL)
if "WEIGHTED_TOTAL" in scorecard:
    ck.record("A8_scorecard", {m: round(float(v), 3) for m, v in scorecard["WEIGHTED_TOTAL"].items()})
print("Requests sent this session (cached replies not counted):", REQUESTS["n"])
ck.record("partA_requests_this_session", REQUESTS["n"])

# Part B (optional stretch, up to 10 bonus points): cost model and build vs. buy

Chapter 4's *Model Build Versus Buy* section compares using a model API with hosting a model yourself. Here you make that concrete with **pretend** numbers (they are invented for the exercise; real prices change constantly).

- **Hosted API** (pretend prices per 1 million tokens, input / output) for models similar to ours.
- **Self-hosted**: a fixed monthly cost (hardware paid off over 3 years, power, and an engineer's time), roughly independent of volume until you run out of capacity.
- The **break-even volume** is the number of questions per month at which the two options cost the same. Below it the API is cheaper; above it, self-hosting is.

Use your measured average input and output tokens per question from A4. Bonus checks are reported separately from the Part A score.

In [ ]:
BONUS = {}


def bonus_check(name, passed, points, hint=""):
    BONUS[name] = points if passed else 0
    print(("✅" if passed else "❌"), f" BONUS {name} ({BONUS[name]}/{points})", "" if passed else f"\n     -> {hint}")


PRETEND_API_PRICES = {            # USD per 1M tokens: (input, output). Invented numbers for this exercise.
    "qwen-small": (0.10, 0.40), "qwen-moe": (0.30, 1.20), "gemma-moe": (0.30, 1.20), "qwen-dense": (0.60, 2.40)}
SELF_HOSTED_MONTHLY = {"hardware_amortized": 12000 / 36, "power": 40.0, "engineer_hours": 6 * 75.0}   # USD per month
tok_stats = runs.groupby("model")[["in_tokens", "out_tokens"]].mean()
display(tok_stats.round(1))

**Reading the table:** the average tokens per question for each model in your A4 run. `in_tokens` is everything sent (system prompt, documents, and question) and `out_tokens` is the answer. Input is much larger because every question carries its documents. The models' tokenizers differ (Lab 01), so the same text can count as a different number of tokens.

In [ ]:
def hosted_monthly_cost(model, questions_per_month, prices=PRETEND_API_PRICES):
    """Pretend API cost per month for `model` at a given volume, using measured mean tokens per question."""
    pin, pout = prices[model]
    tin, tout = tok_stats.loc[model, "in_tokens"], tok_stats.loc[model, "out_tokens"]
    if pd.isna(tin) or pd.isna(tout):
        return None
    return None   # TODO: questions_per_month * (tin * pin + tout * pout) / 1e6


SELF_HOSTED_TOTAL = sum(SELF_HOSTED_MONTHLY.values())


def break_even_volume(model, prices=PRETEND_API_PRICES):
    """Questions per month at which the pretend API costs as much as self-hosting (None if undefined)."""
    return None   # TODO: SELF_HOSTED_TOTAL divided by the hosted cost of ONE question (guard against 0/None)


VOLUMES = [5_000, 50_000, 500_000]   # questions per month
cost_table = pd.DataFrame({v: {m: hosted_monthly_cost(m, v) for m in CANDIDATES} for v in VOLUMES})
cost_table["break_even_q_per_month"] = {m: break_even_volume(m) for m in CANDIDATES}
# TODO: add a column "break_even_if_api_prices_double" (call break_even_volume with every price doubled)
display(cost_table)
print("Self-hosted fixed cost per month:", round(SELF_HOSTED_TOTAL, 2))

BUILD_VS_BUY = """TODO: your build-vs-buy recommendation for Larkspur (at least 4 sentences, using your numbers)."""

**Reading the cost table:** one row per model. The columns `5000`, `50000`, and `500000` are the pretend API cost in dollars per month at that many questions per month. `break_even_q_per_month` is the volume at which the API would cost as much as self-hosting, and `break_even_if_api_prices_double` repeats it with every price doubled. That last column is a **sensitivity check**: it shows how much your conclusion depends on one assumption.

In [ ]:
def _f(x):
    return x is not None and not pd.isna(x)

ref_ok = False
try:
    m0 = list(CANDIDATES)[0]
    tin, tout = tok_stats.loc[m0, "in_tokens"], tok_stats.loc[m0, "out_tokens"]
    expect = 1000 * (tin * PRETEND_API_PRICES[m0][0] + tout * PRETEND_API_PRICES[m0][1]) / 1e6
    got = hosted_monthly_cost(m0, 1000)
    ref_ok = _f(got) and abs(got - expect) < 1e-9
except Exception as e:
    print("hosted_monthly_cost error:", e)
bonus_check("B_hosted_cost", ref_ok, 2, "hosted_monthly_cost = volume * (in_tokens*input_price + out_tokens*output_price) / 1e6")
be = [break_even_volume(m) for m in CANDIDATES]
bonus_check("B_break_even", all(_f(x) and x > 0 for x in be), 2, "break_even_volume should be a positive number for every model.")
bonus_check("B_cost_table", all(_f(x) for x in cost_table[VOLUMES].values.flatten()), 2, "Fill the cost table for all volumes.")
bonus_check("B_sensitivity", "break_even_if_api_prices_double" in cost_table and cost_table["break_even_if_api_prices_double"].notna().all(),
            2, "Add the doubled-price break-even column.")
bonus_check("B_recommendation", "TODO" not in BUILD_VS_BUY and len(BUILD_VS_BUY) >= 300, 2,
            "Write your recommendation in BUILD_VS_BUY (at least 300 characters).")
ck.record("partB_bonus_points", sum(BONUS.values()))
ck.record("partB_bonus_max", 10)

# Written analysis

Use **your numbers**. Cite Chapter 4 by section name where asked. A short paragraph each.

**Q1. Your pick.** Which model did you choose, and what in your comparison table and scorecard supports it? Would a reasonable change in your weights (say, doubling the cost/latency weight) change the winner? Relate your criteria to the chapter's *Evaluation Criteria* (domain-specific capability, generation capability, instruction-following capability, cost and latency).

*Your answer:*

**Q2. Trap questions.** How often did each model follow the outdated document? Describe one case in detail: the two documents, what the model said, and what in the context could have told it which document to trust. Is choosing the document with the latest effective date always enough? Look closely at your trap questions' contexts before answering. What would you change in the system prompt or the context (Lab 7) to fix this?

*Your answer:*

**Q3. Benchmarks vs. your own evaluation.** The chapter's *Model Selection Workflow* runs from filtering by hard attributes, to public information, to your own evaluation, to monitoring in production. Which steps did this lab cover, and which did it skip? Explain *Data contamination with public benchmarks* and why it makes leaderboard scores a weak guide here. Then notice that our gold set lives in a **public** GitHub repository: what does that mean for anyone who evaluates future models on it?

*Your answer:*

**Q4. How much do you trust these numbers?** Name two weaknesses of this evaluation (think: 40 questions, one judge from the same family as one candidate, latency measured on a shared cluster, a regex for abstention) and how you would fix each before Larkspur commits to a model.

*Your answer:*

# AI-use reflection

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. Before you saw the results, which model did you expect to win, and why? Did the numbers change your mind?

*Your answer:*

## Submit

Run this last, save, upload the .ipynb. (Part B bonus points appear in the records as `partB_bonus_points`.)

In [ ]:
ck.report()